Assemble clinical protocols for each recorded Delphi round for subsequent judging.


In [ ]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'experiments/fixed_length_controls/round_evaluation'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import os
import json
import hashlib
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Any, List, Tuple, Optional
import pandas as pd
import requests
from openai import OpenAI
from tqdm.auto import tqdm


In [ ]:
FOLDER_ID = ''
API_KEY = ''
LLM_API_URL = 'https://llm.api.cloud.yandex.net/foundationModels/v1/completion'
INPUT_CSV = '../delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer/llmfmed_dataset.csv'
OUTPUT_CSV = 'delphi_noroles.csv'
CASE_COMPONENT_COLUMNS = ['Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results']
REQUEST_TIMEOUT = 120
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
MAX_DELPHI_ROUNDS = 3
EARLY_STOP_ON_UNANIMITY = True
CASE_CONCURRENCY = 5
QUESTIONS_PER_EXPERT_R1 = 8
MAX_UNIFIED_QUESTIONS = 12
TEMPERATURE_ROUND1 = 0.3
TEMPERATURE_ROUND2 = 0.3
TEMPERATURE_ROUND3 = 0.3
TEMPERATURE_ORGANIZER = 0.3
TEMPERATURE_FINAL_SYNTHESIS = 0.3
MAX_TOKENS_ROUND1 = 10000
MAX_TOKENS_ROUND2 = 10000
MAX_TOKENS_ROUND3 = 10000
MAX_TOKENS_ORGANIZER = 10000
MAX_TOKENS_FINAL_SYNTHESIS = 10000
EMERGENCY_LABELS = ['Self-care appropriate', 'Requires non-emergent care', 'Requires emergent care']
SUBCLINICAL_LABELS = ['Cardiology', 'Dermatology', 'Emergency', 'Endocrinology', 'Gastroenterology', 'General Medicine', 'General Surgery', 'Infectious Disease', 'Nephrology', 'Neurology', 'Obstetrics and Gynecology', 'Ophthalmology', 'Orthopedics', 'Other departments', 'Otolaryngology (ENT)', 'Pediatrics', 'Psychiatry', 'Pulmonology', 'Urology']
PARTICIPANTS: List[Dict[str, str]] = [{'id': 'deepseekv4_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/deepseek-v4-flash/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}, {'id': 'gpt120b_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/gpt-oss-120b/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}, {'id': 'qwen_default', 'provider': 'openai_compatible', 'model_uri': f'gpt://{FOLDER_ID}/qwen3-235b-a22b-fp8/latest', 'role_name': 'Default', 'role_description': 'You are a careful expert assistant. Follow the requested format exactly. Do not use information not grounded in the provided case.'}]
ORGANIZER_PARTICIPANT_ID = 'gpt120b_default'
if ORGANIZER_PARTICIPANT_ID not in [p['id'] for p in PARTICIPANTS]:
    raise ValueError(f'ORGANIZER_PARTICIPANT_ID={ORGANIZER_PARTICIPANT_ID!r} not in PARTICIPANTS ids.')
participant_ids = [p['id'] for p in PARTICIPANTS]
if len(participant_ids) != len(set(participant_ids)):
    raise ValueError('Participant ids must be unique.')
df = pd.read_csv(INPUT_CSV, sep=None, engine='python')
df = df.copy().reset_index(drop=True)
missing_case_columns = [col for col in CASE_COMPONENT_COLUMNS if col not in df.columns]
if missing_case_columns:
    raise ValueError(f'Missing case columns in dataset: {missing_case_columns}')

def build_case_text(row_data: Dict[str, Any]) -> str:
    parts: List[str] = []
    for column in CASE_COMPONENT_COLUMNS:
        value = row_data.get(column)
        if pd.isna(value):
            continue
        value_str = str(value).strip()
        if not value_str or value_str.lower() == 'nan':
            continue
        parts.append(f'{column}:\n{value_str}')
    return '\n\n'.join(parts)
df['case_text'] = df.apply(lambda row: build_case_text(row.to_dict()), axis=1)
print('Rows:', len(df))
print('Columns:', df.columns.tolist())
print('Participants (experts):', [p['id'] for p in PARTICIPANTS])
print('Organizer:', ORGANIZER_PARTICIPANT_ID)
print('Case columns:', CASE_COMPONENT_COLUMNS)


In [ ]:
session = requests.Session()
llm_cache: Dict[str, Dict[str, Any]] = {}
openai_client = OpenAI(api_key=API_KEY, base_url='https://llm.api.cloud.yandex.net/v1', project=FOLDER_ID)

def make_cache_key(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> str:
    payload = {'provider': provider, 'model_uri': model_uri, 'system_text': system_text, 'user_text': user_text, 'temperature': temperature, 'max_tokens': max_tokens}
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _build_token_dict(prompt_tokens: int=0, completion_tokens: int=0, total_tokens: int=0) -> Dict[str, int]:
    return {'prompt_tokens': int(prompt_tokens or 0), 'completion_tokens': int(completion_tokens or 0), 'total_tokens': int(total_tokens or 0)}

def _extract_openai_text(response: Any) -> str:
    if not getattr(response, 'choices', None):
        return ''
    message = response.choices[0].message
    content = getattr(message, 'content', '')
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
            elif hasattr(item, 'type') and getattr(item, 'type', None) == 'text':
                parts.append(getattr(item, 'text', ''))
        return '\n'.join([p for p in parts if p]).strip()
    return ''

def _extract_openai_usage(usage: Any) -> Dict[str, int]:
    if usage is None:
        return _build_token_dict()
    prompt_tokens = getattr(usage, 'prompt_tokens', None)
    completion_tokens = getattr(usage, 'completion_tokens', None)
    total_tokens = getattr(usage, 'total_tokens', None)
    if total_tokens is None:
        input_tokens = getattr(usage, 'input_tokens', None)
        output_tokens = getattr(usage, 'output_tokens', None)
        prompt_tokens = prompt_tokens if prompt_tokens is not None else input_tokens
        completion_tokens = completion_tokens if completion_tokens is not None else output_tokens
        total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def _extract_yandex_usage(result: Dict[str, Any]) -> Dict[str, int]:
    usage = result.get('usage', {}) or {}
    prompt_tokens = usage.get('inputTextTokens', 0) or 0
    completion_tokens = usage.get('completionTokens', 0) or 0
    total_tokens = usage.get('totalTokens', 0) or prompt_tokens + completion_tokens
    return _build_token_dict(prompt_tokens, completion_tokens, total_tokens)

def call_yandex_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, timeout: int=REQUEST_TIMEOUT, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('yandex_rest', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        cached = llm_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    headers = {'Content-Type': 'application/json', 'Authorization': f'Api-Key {API_KEY}'}
    payload = {'modelUri': model_uri, 'completionOptions': {'stream': False, 'temperature': temperature, 'maxTokens': max_tokens}, 'messages': [{'role': 'system', 'text': system_text}, {'role': 'user', 'text': user_text}]}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = session.post(LLM_API_URL, headers=headers, json=payload, timeout=timeout)
        except requests.RequestException as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        if response.status_code == 200:
            data = response.json()
            result = data.get('result', {}) or {}
            raw_text = result.get('alternatives', [{}])[0].get('message', {}).get('text', '').strip()
            usage = _extract_yandex_usage(result)
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, 'prompt_tokens': usage['prompt_tokens'], 'completion_tokens': usage['completion_tokens'], 'total_tokens': usage['total_tokens'], 'cached': False}
        last_error_text = f'HTTP {response.status_code}: {response.text}'
        if response.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
        raise RuntimeError(last_error_text)
    raise RuntimeError(f'Yandex REST call failed after {retries} retries. Last error: {last_error_text}')

def call_openai_llm(model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int, retries: int=MAX_RETRIES) -> Dict[str, Any]:
    cache_key = make_cache_key('openai_compatible', model_uri, system_text, user_text, temperature, max_tokens)
    if cache_key in llm_cache:
        cached = llm_cache[cache_key]
        return {'text': cached['text'], 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0, 'cached': True}
    last_error_text = ''
    for attempt in range(retries):
        try:
            response = openai_client.chat.completions.create(model=model_uri, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens)
            raw_text = _extract_openai_text(response)
            usage = _extract_openai_usage(getattr(response, 'usage', None))
            llm_cache[cache_key] = {'text': raw_text}
            return {'text': raw_text, 'prompt_tokens': usage['prompt_tokens'], 'completion_tokens': usage['completion_tokens'], 'total_tokens': usage['total_tokens'], 'cached': False}
        except Exception as e:
            last_error_text = str(e)
            time.sleep(BASE_SLEEP_SECONDS * 2 ** attempt)
            continue
    raise RuntimeError(f'OpenAI-compatible call failed after {retries} retries. Last error: {last_error_text}')

def call_llm(provider: str, model_uri: str, system_text: str, user_text: str, temperature: float, max_tokens: int) -> Dict[str, Any]:

    def _dispatch() -> Dict[str, Any]:
        if provider == 'yandex_rest':
            return call_yandex_llm(model_uri, system_text, user_text, temperature, max_tokens)
        if provider == 'openai_compatible':
            return call_openai_llm(model_uri, system_text, user_text, temperature, max_tokens)
        raise ValueError(f'Unknown provider: {provider}')
    response = _dispatch()
    if str(response['text']).strip():
        return response
    llm_cache.pop(make_cache_key(provider, model_uri, system_text, user_text, temperature, max_tokens), None)
    retry = _dispatch()
    for key in ('prompt_tokens', 'completion_tokens', 'total_tokens'):
        retry[key] += response[key]
    if not str(retry['text']).strip():
        print(f'WARNING: empty response after retry (model_uri={model_uri}, max_tokens={max_tokens})')
    return retry


In [ ]:
def build_round1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are one of the expert physicians participating anonymously in a Delphi method process.\nAt round 1 your task is to decompose the diagnostic and management problem implied by a clinical case\ninto a set of focused sub-questions, contribute extra clinical considerations you believe are relevant\nbeyond what is explicitly stated, and provide your initial answers.\n\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\nReturn JSON only.'.strip()

def build_round1_user_prompt(case_text: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nTasks:\n1) Decompose the case into exactly {QUESTIONS_PER_EXPERT_R1} sub-questions that\n   together cover diagnosis, urgency, specialty routing, differential, red flags, diagnostics,\n   and management. Each sub-question must be answerable from the case content.\n2) For each sub-question give your initial answer and a brief rationale.\n3) List additional clinical considerations you think are relevant but were not explicitly highlighted in\n   the case (e.g. red flags, dangerous differentials, ambiguous findings).\n4) Give your initial classification (Emergency type, Subclinical specialization, confidence).\n   Confidence is your certainty in the two labels you just chose, rated 1-10. Rate 1-3 if the case\n   under-determines the labels, 4-7 if they are the best fit among defensible alternatives, and 8-10\n   if the case explicitly determines them.\n\nReturn JSON only:\n{{\n  "sub_questions": [\n    {{"id": "Q1", "question": "...", "my_answer": "...", "rationale": "..."}}\n  ],\n  "additional_relevant_facts": ["..."],\n  "initial_classification": {{\n    "emergency_type": "<one of {emergency_list}>",\n    "subclinical_specialization": "<one of {subclinical_list}>",\n    "confidence": 0\n  }}\n}}\n'.strip()

def build_organizer_compile_after_r1_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi process moderator. The anonymous experts have just completed round 1\nand produced their decompositions and initial answers.\n\nTasks:\n1) Deduplicate and merge the sub-questions into a single unified questionnaire of at most\n   {MAX_UNIFIED_QUESTIONS} items. Prefer sub-questions raised by more experts; keep clinically critical\n   ones even if raised by only one expert.\n2) For each unified question, produce a short anonymous summary of how experts answered, and explicitly\n   list any extreme/outlier positions with their stated reasons.\n3) Preserve all additional clinical considerations contributed by experts.\n\nReturn JSON only.'.strip()

def build_organizer_compile_after_r1_user_prompt(case_text: str, expert_payloads: str) -> str:
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nRound-1 raw expert payloads (anonymized as Expert A, Expert B, ...):\n<<<EXPERT_PAYLOADS>>>\n{expert_payloads}\n<<<END EXPERT_PAYLOADS>>>\n\nReturn JSON only in this exact format:\n{{\n  "unified_questions": [\n    {{\n      "id": "U1",\n      "question": "...",\n      "anonymous_summary_of_answers": "1-3 sentence summary of how experts answered",\n      "extreme_views": ["each outlier position with its stated reason"]\n    }}\n  ],\n  "merged_additional_considerations": ["..."],\n  "anonymous_overall_summary": "1 paragraph"\n}}\n\nHard constraints:\n1) Number of unified_questions in [5, {MAX_UNIFIED_QUESTIONS}].\n2) Output JSON only.'.strip()

def build_round_n_system_prompt(role_name: str, role_description: str, round_idx: int) -> str:
    return f"You are acting as {role_name}.\n{role_description}\n\nYou are one of the expert physicians participating anonymously in a Delphi method process.\nThis is round {round_idx} of {MAX_DELPHI_ROUNDS}.\n\nYou will see a unified questionnaire compiled from all experts' previous-round answers, with an\nanonymous summary of others' positions and any extreme views with their reasons. Your task is to\nre-answer each question, indicate whether your answer has changed since the previous round and why,\nand update your classification.\n\nBe willing to revise. Be willing to keep your position if the case supports it. Return JSON only.".strip()

def build_round2_user_prompt(case_text: str, organizer_payload: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nUnified questionnaire and anonymous summary from round 1:\n<<<ORGANIZER_SUMMARY>>>\n{organizer_payload}\n<<<END ORGANIZER_SUMMARY>>>\n\nTasks:\n1) Answer each unified question. State whether your answer changed compared to your round-1 position\n   ("changed_from_previous": true/false) and explain why in one short sentence.\n2) Indicate whether the unified questionnaire is sufficient (covers everything important for this case).\n3) If not sufficient, list missing aspects that should be added in round 3.\n4) Update your classification (Emergency type, Subclinical specialization, confidence).\n   Confidence is your certainty in the two labels you just chose, rated 1-10. Rate 1-3 if the case\n   under-determines the labels, 4-7 if they are the best fit among defensible alternatives, and 8-10\n   if the case explicitly determines them.\n\nReturn JSON only:\n{{\n  "answers": [\n    {{\n      "question_id": "U1",\n      "answer": "...",\n      "rationale": "...",\n      "changed_from_previous": true\n    }}\n  ],\n  "is_questionnaire_sufficient": true,\n  "missing_aspects": ["..."],\n  "classification": {{\n    "emergency_type": "<one of {emergency_list}>",\n    "subclinical_specialization": "<one of {subclinical_list}>",\n    "confidence": 0\n  }}\n}}\n'.strip()

def build_organizer_compile_after_r2_system_prompt(role_name: str, role_description: str) -> str:
    return f'You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi process moderator. The anonymous experts have just completed round 2.\n\nTasks:\n1) Produce an anonymous summary of round-2 answers per unified question.\n2) Explicitly highlight every outlier / extreme view (a position not shared by majority) with the reason\n   the dissenting expert provided. This is what experts must address in round 3.\n3) Carry forward any missing aspects that experts flagged. You may add additional questions if\n   experts unanimously requested them.\n\nReturn JSON only.'.strip()

def build_organizer_compile_after_r2_user_prompt(case_text: str, organizer_summary_after_r1: str, expert_r2_payloads: str) -> str:
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nOrganizer summary from round 1 (for continuity):\n<<<R1_SUMMARY>>>\n{organizer_summary_after_r1}\n<<<END R1_SUMMARY>>>\n\nRound-2 raw expert payloads (anonymized as Expert A, Expert B, ...):\n<<<R2_PAYLOADS>>>\n{expert_r2_payloads}\n<<<END R2_PAYLOADS>>>\n\nReturn JSON only:\n{{\n  "unified_questions_for_round3": [\n    {{\n      "id": "U1",\n      "question": "...",\n      "anonymous_summary_of_round2_answers": "...",\n      "extreme_views_with_reasons": ["explicit outlier-position + stated reason"]\n    }}\n  ],\n  "added_questions": ["any new questions, if all experts requested them"],\n  "anonymous_overall_summary": "1 paragraph"\n}}\n\nHard constraints:\n1) Preserve all original unified question ids unless an expert majority requested removal.\n2) Output JSON only.'.strip()

def build_round3_user_prompt(case_text: str, organizer_payload: str) -> str:
    emergency_list = json.dumps(EMERGENCY_LABELS)
    subclinical_list = json.dumps(SUBCLINICAL_LABELS)
    return f"""Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nUnified questionnaire from round 2, with anonymous summary and explicit extreme views with reasons:\n<<<ORGANIZER_SUMMARY>>>\n{organizer_payload}\n<<<END ORGANIZER_SUMMARY>>>\n\nTasks:\n1) Provide your final answer to each question; mark whether you changed your view ("changed_from_previous").\n2) Explicitly review extreme views — list which ones you considered and whether they updated your stance.\n3) Rate the case on three Delphi axes, each 1-10:\n   - "efficiency_of_intervention": expected efficiency of the recommended management.\n   - "resource_adequacy": adequacy of resources/diagnostics implied by the recommendation.\n   - "alignment_with_original_problem": match between your recommended plan and the case's core problem.\n4) Finalize your classification (Emergency type, Subclinical specialization, confidence).\n   Confidence is your certainty in the two labels you just chose, rated 1-10 on the same scale as the\n   Delphi axes above. Rate 1-3 if the case under-determines the labels, 4-7 if they are the best fit\n   among defensible alternatives, and 8-10 if the case explicitly determines them.\n\nReturn JSON only:\n{{\n  "answers": [\n    {{\n      "question_id": "U1",\n      "answer": "...",\n      "rationale": "...",\n      "changed_from_previous": false\n    }}\n  ],\n  "outliers_addressed": [\n    {{"summary_of_outlier": "...", "updated_my_view": false, "reason": "..."}}\n  ],\n  "ratings": {{\n    "efficiency_of_intervention": 0,\n    "resource_adequacy": 0,\n    "alignment_with_original_problem": 0\n  }},\n  "classification": {{\n    "emergency_type": "<one of {emergency_list}>",\n    "subclinical_specialization": "<one of {subclinical_list}>",\n    "confidence": 0\n  }}\n}}\n""".strip()

def build_final_synthesis_system_prompt(role_name: str, role_description: str) -> str:
    return f"""You are acting as {role_name}.\n{role_description}\n\nYou are the Delphi moderator. The expert panel has converged (or reached the round limit). You receive\nthe unified questionnaire, all experts' final answers, their ratings, and the consensus labels for\nEmergency type and Subclinical specialization (already computed as the majority of expert votes).\n\nCompose a single clinical protocol that uses the consensus labels exactly, integrates the supplied\nanswers into the Complaint, Analysis, Required diagnostic equipment/tests, Conclusion, Recommendations,\nNotes sections, and explicitly addresses any considerations and divergences that surfaced in the\nmaterial below.\n\nWrite the protocol as a standalone clinical document. Do not describe the process by which the\nconclusions were reached; state them directly.\n\nUse only information explicitly present in the case. If a detail is absent, write "Not specified"; if a field is unnecessary, write "None".\nReason carefully from the case. Do not invent symptoms or findings absent from the case.\n\nLength requirement: the complete protocol must contain between 400 and 500 words in total,\ncounting all fields together. Protocols shorter than 400 or longer than 500 words are invalid.""".strip()

def build_final_synthesis_user_prompt(case_text: str, consensus_emergency: str, consensus_subclinical: str, rounds_summary: str, final_answers_block: str, ratings_block: str, additional_considerations_block: str, author_id: str) -> str:
    return f'Clinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nConsensus labels (majority of final-round expert votes; use these verbatim):\n- Emergency type: {consensus_emergency}\n- Subclinical specialization: {consensus_subclinical}\n\nCase discussion summary:\n<<<ROUNDS_SUMMARY>>>\n{rounds_summary}\n<<<END ROUNDS_SUMMARY>>>\n\nAnswers per unified question:\n<<<FINAL_ANSWERS>>>\n{final_answers_block}\n<<<END FINAL_ANSWERS>>>\n\nRatings (efficiency / resource_adequacy / alignment):\n<<<RATINGS>>>\n{ratings_block}\n<<<END RATINGS>>>\n\nAdditional considerations surfaced during the rounds:\n<<<EXTRA>>>\n{additional_considerations_block}\n<<<END EXTRA>>>\n\nWrite the protocol in plain text. Same field labels and order:\nTotal length: 400-500 words across all fields.\n\nAge:\nSex:\nComplaint:\nAnalysis:\nRequired diagnostic equipment/tests:\nConclusion:\nEmergency type:\nSubclinical specialization:\nRecommendations:\nNotes:\nProtocol Author: {author_id}\n\nField instructions:\nAge: copy from the case if stated; otherwise write "Not specified".\nSex: copy from the case if stated; otherwise write "Not specified".\nComplaint: briefly summarize the presenting problem from the case.\nAnalysis: explain the clinical reasoning, the strongest supporting facts, and the main relevant differential considerations.\nRequired diagnostic equipment/tests: mention only studies or equipment that are actually needed next; if nothing additional is needed, write "None".\nConclusion: give a concise clinical synthesis on the case.\nEmergency type: based on your analysis of this case, emergency type must equal exactly: {consensus_emergency}\nSubclinical specialization: based on your analysis of this case, subclinical specialization must equal exactly: {consensus_subclinical}\nRecommendations: provide practical, case-grounded next steps or management advice. Explicitly address the additional considerations.\nNotes: include only essential caveats; otherwise write "None".\n'.strip()

def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    if text is None:
        return None
    raw = str(text).strip()
    try:
        return json.loads(raw)
    except Exception:
        pass
    match = re.search('\\{.*\\}', raw, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None

def _coerce_str(value: Any) -> str:
    if value is None:
        return ''
    return str(value).strip()

def _normalize_label(value: Any, allowed: List[str]) -> str:
    if value is None:
        return ''
    s = str(value).strip()
    if not s:
        return ''
    lookup = {a.lower(): a for a in allowed}
    return lookup.get(s.lower(), '')

def _int_in_range(value: Any, lo: int, hi: int, default: int) -> int:
    try:
        v = int(value)
    except Exception:
        return default
    return max(lo, min(hi, v))

def parse_round1_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    sub_qs_raw = data.get('sub_questions') or []
    sub_qs: List[Dict[str, str]] = []
    if isinstance(sub_qs_raw, list):
        for idx, item in enumerate(sub_qs_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'Q{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            sub_qs.append({'id': qid, 'question': question, 'my_answer': _coerce_str(item.get('my_answer')), 'rationale': _coerce_str(item.get('rationale'))})
    facts_raw = data.get('additional_relevant_facts') or []
    if not isinstance(facts_raw, list):
        facts_raw = [_coerce_str(facts_raw)]
    facts = [_coerce_str(x) for x in facts_raw if _coerce_str(x)]
    cls = data.get('initial_classification') or {}
    return {'sub_questions': sub_qs, 'additional_relevant_facts': facts, 'classification': {'emergency_type': _normalize_label(cls.get('emergency_type'), EMERGENCY_LABELS), 'subclinical': _normalize_label(cls.get('subclinical_specialization'), SUBCLINICAL_LABELS), 'confidence': _int_in_range(cls.get('confidence'), 1, 10, 5)}, 'raw_text': text}

def parse_compile_after_r1_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    unified_raw = data.get('unified_questions') or []
    unified: List[Dict[str, Any]] = []
    if isinstance(unified_raw, list):
        for idx, item in enumerate(unified_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'U{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            extremes = item.get('extreme_views') or []
            if not isinstance(extremes, list):
                extremes = [_coerce_str(extremes)]
            extremes = [_coerce_str(x) for x in extremes if _coerce_str(x)]
            unified.append({'id': qid, 'question': question, 'anonymous_summary_of_answers': _coerce_str(item.get('anonymous_summary_of_answers')), 'extreme_views': extremes})
    extras_raw = data.get('merged_additional_considerations') or []
    if not isinstance(extras_raw, list):
        extras_raw = [_coerce_str(extras_raw)]
    extras = [_coerce_str(x) for x in extras_raw if _coerce_str(x)]
    return {'unified_questions': unified, 'merged_additional_considerations': extras, 'anonymous_overall_summary': _coerce_str(data.get('anonymous_overall_summary')), 'raw_text': text}

def parse_round_n_output(text: str, round_idx: int) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    answers_raw = data.get('answers') or []
    answers: List[Dict[str, Any]] = []
    if isinstance(answers_raw, list):
        for item in answers_raw:
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('question_id'))
            if not qid:
                continue
            answers.append({'question_id': qid, 'answer': _coerce_str(item.get('answer')), 'rationale': _coerce_str(item.get('rationale')), 'changed_from_previous': bool(item.get('changed_from_previous'))})
    missing_raw = data.get('missing_aspects') or []
    if not isinstance(missing_raw, list):
        missing_raw = [_coerce_str(missing_raw)]
    missing = [_coerce_str(x) for x in missing_raw if _coerce_str(x)]
    cls = data.get('classification') or {}
    return {'round_idx': round_idx, 'answers': answers, 'is_questionnaire_sufficient': bool(data.get('is_questionnaire_sufficient', True)), 'missing_aspects': missing, 'classification': {'emergency_type': _normalize_label(cls.get('emergency_type'), EMERGENCY_LABELS), 'subclinical': _normalize_label(cls.get('subclinical_specialization'), SUBCLINICAL_LABELS), 'confidence': _int_in_range(cls.get('confidence'), 1, 10, 5)}, 'raw_text': text}

def parse_compile_after_r2_output(text: str) -> Dict[str, Any]:
    data = extract_first_json_object(text) or {}
    unified_raw = data.get('unified_questions_for_round3') or []
    unified: List[Dict[str, Any]] = []
    if isinstance(unified_raw, list):
        for idx, item in enumerate(unified_raw, start=1):
            if not isinstance(item, dict):
                continue
            qid = _coerce_str(item.get('id')) or f'U{idx}'
            question = _coerce_str(item.get('question'))
            if not question:
                continue
            extremes = item.get('extreme_views_with_reasons') or []
            if not isinstance(extremes, list):
                extremes = [_coerce_str(extremes)]
            extremes = [_coerce_str(x) for x in extremes if _coerce_str(x)]
            unified.append({'id': qid, 'question': question, 'anonymous_summary_of_round2_answers': _coerce_str(item.get('anonymous_summary_of_round2_answers')), 'extreme_views_with_reasons': extremes})
    added_raw = data.get('added_questions') or []
    if not isinstance(added_raw, list):
        added_raw = [_coerce_str(added_raw)]
    added = [_coerce_str(x) for x in added_raw if _coerce_str(x)]
    return {'unified_questions_for_round3': unified, 'added_questions': added, 'anonymous_overall_summary': _coerce_str(data.get('anonymous_overall_summary')), 'raw_text': text}

def parse_round3_output(text: str) -> Dict[str, Any]:
    base = parse_round_n_output(text, 3)
    data = extract_first_json_object(text) or {}
    outliers_raw = data.get('outliers_addressed') or []
    outliers: List[Dict[str, Any]] = []
    if isinstance(outliers_raw, list):
        for item in outliers_raw:
            if not isinstance(item, dict):
                continue
            outliers.append({'summary_of_outlier': _coerce_str(item.get('summary_of_outlier')), 'updated_my_view': bool(item.get('updated_my_view')), 'reason': _coerce_str(item.get('reason'))})
    ratings_raw = data.get('ratings') or {}
    ratings = {'efficiency_of_intervention': _int_in_range(ratings_raw.get('efficiency_of_intervention'), 1, 10, 5), 'resource_adequacy': _int_in_range(ratings_raw.get('resource_adequacy'), 1, 10, 5), 'alignment_with_original_problem': _int_in_range(ratings_raw.get('alignment_with_original_problem'), 1, 10, 5)}
    base['outliers_addressed'] = outliers
    base['ratings'] = ratings
    return base

def extract_field_value(text: str, field: str) -> str:
    if text is None:
        return ''
    m = re.search(f'{re.escape(field)}\\s*:\\s*(.+)', str(text), re.IGNORECASE)
    return m.group(1).strip() if m else ''

def extract_emergency_type(protocol_text: str) -> str:
    return _normalize_label(extract_field_value(protocol_text, 'Emergency type'), EMERGENCY_LABELS)

def extract_subclinical(protocol_text: str) -> str:
    return _normalize_label(extract_field_value(protocol_text, 'Subclinical specialization'), SUBCLINICAL_LABELS)


In [ ]:
participant_order = {p['id']: i for i, p in enumerate(PARTICIPANTS)}

def _format_anonymous_label(idx: int) -> str:
    return f"Expert {chr(ord('A') + idx)}"

def _format_round1_payloads(round1_outputs: List[Dict[str, Any]]) -> str:
    chunks: List[str] = []
    for idx, out in enumerate(round1_outputs):
        chunks.append(f'### {_format_anonymous_label(idx)}')
        chunks.append('Initial classification:')
        cls = out['parsed']['classification']
        chunks.append(f"  emergency_type: {cls['emergency_type']}")
        chunks.append(f"  subclinical: {cls['subclinical']}")
        chunks.append(f"  confidence: {cls['confidence']}")
        chunks.append('Sub-questions and initial answers:')
        for q in out['parsed']['sub_questions']:
            chunks.append(f"  [{q['id']}] Q: {q['question']}")
            chunks.append(f"      A: {q['my_answer']}")
            if q['rationale']:
                chunks.append(f"      rationale: {q['rationale']}")
        if out['parsed']['additional_relevant_facts']:
            chunks.append('Additional considerations:')
            for f in out['parsed']['additional_relevant_facts']:
                chunks.append(f'  - {f}')
        chunks.append('')
    return '\n'.join(chunks).strip()

def _format_round_n_payloads(round_outputs: List[Dict[str, Any]], round_idx: int) -> str:
    chunks: List[str] = []
    for idx, out in enumerate(round_outputs):
        chunks.append(f'### {_format_anonymous_label(idx)}')
        cls = out['parsed'].get('classification') or {'emergency_type': '', 'subclinical': '', 'confidence': 0}
        chunks.append('Classification at this round:')
        chunks.append(f"  emergency_type: {cls.get('emergency_type', '')}")
        chunks.append(f"  subclinical: {cls.get('subclinical', '')}")
        chunks.append(f"  confidence: {cls.get('confidence', 0)}")
        chunks.append(f"is_questionnaire_sufficient: {out['parsed'].get('is_questionnaire_sufficient', True)}")
        if out['parsed'].get('missing_aspects'):
            chunks.append('missing_aspects:')
            for m in out['parsed']['missing_aspects']:
                chunks.append(f'  - {m}')
        chunks.append('Answers:')
        for a in out['parsed'].get('answers', []):
            chunks.append(f"  [{a['question_id']}] A: {a['answer']}")
            if a['rationale']:
                chunks.append(f"      rationale: {a['rationale']}")
            chunks.append(f"      changed_from_previous: {a['changed_from_previous']}")
        if round_idx == 3 and 'ratings' in out['parsed']:
            r = out['parsed']['ratings']
            chunks.append('Ratings:')
            chunks.append(f"  efficiency_of_intervention: {r['efficiency_of_intervention']}")
            chunks.append(f"  resource_adequacy: {r['resource_adequacy']}")
            chunks.append(f"  alignment_with_original_problem: {r['alignment_with_original_problem']}")
            if out['parsed'].get('outliers_addressed'):
                chunks.append('outliers_addressed:')
                for o in out['parsed']['outliers_addressed']:
                    chunks.append(f"  - outlier: {o['summary_of_outlier']}; updated_my_view: {o['updated_my_view']}; reason: {o['reason']}")
        chunks.append('')
    return '\n'.join(chunks).strip()

def _format_organizer_payload_for_expert(compiled: Dict[str, Any], variant: str) -> str:
    chunks: List[str] = []
    if variant == 'after_r1':
        chunks.append('Unified questionnaire (round 2 input):')
        for q in compiled.get('unified_questions', []):
            chunks.append(f"  [{q['id']}] {q['question']}")
            if q['anonymous_summary_of_answers']:
                chunks.append(f"      anonymous summary: {q['anonymous_summary_of_answers']}")
            if q['extreme_views']:
                chunks.append('      extreme views:')
                for ev in q['extreme_views']:
                    chunks.append(f'        * {ev}')
        if compiled.get('merged_additional_considerations'):
            chunks.append('Additional considerations to keep in mind:')
            for c in compiled['merged_additional_considerations']:
                chunks.append(f'  - {c}')
        if compiled.get('anonymous_overall_summary'):
            chunks.append(f"Overall summary: {compiled['anonymous_overall_summary']}")
    else:
        chunks.append('Unified questionnaire (round 3 input) with extreme views:')
        for q in compiled.get('unified_questions_for_round3', []):
            chunks.append(f"  [{q['id']}] {q['question']}")
            if q['anonymous_summary_of_round2_answers']:
                chunks.append(f"      anonymous summary: {q['anonymous_summary_of_round2_answers']}")
            if q['extreme_views_with_reasons']:
                chunks.append('      extreme views with reasons:')
                for ev in q['extreme_views_with_reasons']:
                    chunks.append(f'        * {ev}')
        if compiled.get('added_questions'):
            chunks.append('Added questions (unanimously requested):')
            for q in compiled['added_questions']:
                chunks.append(f'  - {q}')
        if compiled.get('anonymous_overall_summary'):
            chunks.append(f"Overall summary: {compiled['anonymous_overall_summary']}")
    return '\n'.join(chunks)

def run_round1(case_text: str, case_participants: List[Dict[str, str]]) -> List[Dict[str, Any]]:

    def _call_one(participant: Dict[str, str]) -> Dict[str, Any]:
        system_text = build_round1_system_prompt(participant['role_name'], participant['role_description'])
        user_text = build_round1_user_prompt(case_text)
        response = call_llm(provider=participant['provider'], model_uri=participant['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ROUND1, max_tokens=MAX_TOKENS_ROUND1)
        print('DELPHI R1 EXPERT:', participant['id'])
        parsed = parse_round1_output(response['text'])
        return {'participant_id': participant['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'parsed': parsed, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}
    with ThreadPoolExecutor(max_workers=len(case_participants)) as pool:
        outputs = list(pool.map(_call_one, case_participants))
    return outputs

def run_organizer_compile_after_r1(case_text: str, round1_outputs: List[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    expert_payloads = _format_round1_payloads(round1_outputs)
    system_text = build_organizer_compile_after_r1_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_organizer_compile_after_r1_user_prompt(case_text, expert_payloads)
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ORGANIZER, max_tokens=MAX_TOKENS_ORGANIZER)
    print('DELPHI ORGANIZER COMPILE AFTER R1')
    compiled = parse_compile_after_r1_output(response['text'])
    return {'organizer_id': organizer['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'compiled': compiled, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def run_round_n(case_text: str, organizer_payload: str, round_idx: int, case_participants: List[Dict[str, str]]) -> List[Dict[str, Any]]:

    def _call_one(participant: Dict[str, str]) -> Dict[str, Any]:
        system_text = build_round_n_system_prompt(participant['role_name'], participant['role_description'], round_idx)
        if round_idx == 2:
            user_text = build_round2_user_prompt(case_text, organizer_payload)
            max_tokens = MAX_TOKENS_ROUND2
            temp = TEMPERATURE_ROUND2
        elif round_idx == 3:
            user_text = build_round3_user_prompt(case_text, organizer_payload)
            max_tokens = MAX_TOKENS_ROUND3
            temp = TEMPERATURE_ROUND3
        else:
            raise ValueError(f'Unsupported round_idx {round_idx} for run_round_n')
        response = call_llm(provider=participant['provider'], model_uri=participant['model_uri'], system_text=system_text, user_text=user_text, temperature=temp, max_tokens=max_tokens)
        print(f'DELPHI R{round_idx} EXPERT:', participant['id'])
        if round_idx == 3:
            parsed = parse_round3_output(response['text'])
        else:
            parsed = parse_round_n_output(response['text'], round_idx)
        return {'participant_id': participant['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'parsed': parsed, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}
    with ThreadPoolExecutor(max_workers=len(case_participants)) as pool:
        outputs = list(pool.map(_call_one, case_participants))
    return outputs

def run_organizer_compile_after_r2(case_text: str, organizer_after_r1_compiled: Dict[str, Any], round2_outputs: List[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    r1_summary_text = _format_organizer_payload_for_expert(organizer_after_r1_compiled, 'after_r1')
    r2_payloads = _format_round_n_payloads(round2_outputs, 2)
    system_text = build_organizer_compile_after_r2_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_organizer_compile_after_r2_user_prompt(case_text, r1_summary_text, r2_payloads)
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_ORGANIZER, max_tokens=MAX_TOKENS_ORGANIZER)
    print('DELPHI ORGANIZER COMPILE AFTER R2')
    compiled = parse_compile_after_r2_output(response['text'])
    return {'organizer_id': organizer['id'], 'system_prompt': system_text, 'user_prompt': user_text, 'raw_text': response['text'], 'compiled': compiled, 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def _classifications_unanimous(per_expert: List[Dict[str, Any]]) -> bool:
    emergencies = {e['classification']['emergency_type'] for e in per_expert if e['classification']['emergency_type']}
    subs = {e['classification']['subclinical'] for e in per_expert if e['classification']['subclinical']}
    return len(emergencies) == 1 and len(subs) == 1

def compute_delphi_consensus(final_round_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    classifications = [o['parsed']['classification'] for o in final_round_outputs]
    emergency_counts = Counter((c['emergency_type'] for c in classifications if c['emergency_type']))
    subclinical_counts = Counter((c['subclinical'] for c in classifications if c['subclinical']))
    moderator_classification = next((o['parsed']['classification'] for o in final_round_outputs if o['participant_id'] == ORGANIZER_PARTICIPANT_ID), None)

    def _pick(counts: Counter, field: str, fallback: str) -> str:
        if not counts:
            return fallback
        top_count = counts.most_common(1)[0][1]
        tied = [label for label, c in counts.items() if c == top_count]
        if len(tied) == 1:
            return tied[0]
        moderator_label = moderator_classification.get(field) if moderator_classification else None
        if moderator_label in tied:
            return moderator_label
        confidence_by_label = {label: 0.0 for label in tied}
        for c in classifications:
            label = c.get(field)
            if label in confidence_by_label:
                conf = c.get('confidence')
                if isinstance(conf, (int, float)):
                    confidence_by_label[label] += float(conf)
        return max(tied, key=lambda label: confidence_by_label[label])
    consensus_emergency = _pick(emergency_counts, 'emergency_type', 'Requires non-emergent care')
    consensus_subclinical = _pick(subclinical_counts, 'subclinical', 'General Medicine')
    return {'consensus_emergency_type': consensus_emergency, 'consensus_subclinical': consensus_subclinical, 'emergency_counts': dict(emergency_counts), 'subclinical_counts': dict(subclinical_counts)}

def _aggregate_ratings(round3_outputs: List[Dict[str, Any]]) -> Dict[str, float]:
    if not round3_outputs:
        return {}
    keys = ['efficiency_of_intervention', 'resource_adequacy', 'alignment_with_original_problem']
    out: Dict[str, float] = {}
    for k in keys:
        values = []
        for o in round3_outputs:
            r = o['parsed'].get('ratings') or {}
            v = r.get(k)
            if isinstance(v, (int, float)):
                values.append(float(v))
        out[k + '_mean'] = round(sum(values) / len(values), 2) if values else 0.0
        out[k + '_median'] = round(sorted(values)[len(values) // 2], 2) if values else 0.0
    return out

def _r1_outputs_as_answer_shape(round1_outputs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    converted: List[Dict[str, Any]] = []
    for o in round1_outputs:
        parsed = o['parsed']
        answers = [{'question_id': q['id'], 'answer': q['my_answer'], 'rationale': q['rationale'], 'changed_from_previous': False} for q in parsed.get('sub_questions', [])]
        converted.append({'participant_id': o['participant_id'], 'parsed': {'answers': answers, 'classification': parsed['classification'], 'is_questionnaire_sufficient': True, 'missing_aspects': []}})
    return converted

def _format_final_answers_block(final_outputs: List[Dict[str, Any]]) -> str:
    return _format_round_n_payloads(final_outputs, 3)

def _format_ratings_block(final_outputs: List[Dict[str, Any]], aggregate: Dict[str, float]) -> str:
    lines: List[str] = []
    for idx, o in enumerate(final_outputs):
        r = o['parsed'].get('ratings') or {}
        lines.append(f"  {_format_anonymous_label(idx)}: efficiency={r.get('efficiency_of_intervention')}, resource_adequacy={r.get('resource_adequacy')}, alignment={r.get('alignment_with_original_problem')}")
    if aggregate:
        lines.append('  Aggregate (mean / median):')
        for k in ['efficiency_of_intervention', 'resource_adequacy', 'alignment_with_original_problem']:
            lines.append(f"    {k}: mean={aggregate.get(k + '_mean')}, median={aggregate.get(k + '_median')}")
    return '\n'.join(lines)

def _format_rounds_summary(organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]]) -> str:
    chunks: List[str] = []
    chunks.append('Round 1 organizer summary:')
    chunks.append(organizer_after_r1['compiled'].get('anonymous_overall_summary', '(empty)'))
    if organizer_after_r2 is not None:
        chunks.append('')
        chunks.append('Round 2 organizer summary:')
        chunks.append(organizer_after_r2['compiled'].get('anonymous_overall_summary', '(empty)'))
    return '\n'.join(chunks)

def _format_additional_considerations(organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]]) -> str:
    items: List[str] = []
    for source in (organizer_after_r1, organizer_after_r2):
        if source is None:
            continue
        compiled = source['compiled']
        for c in compiled.get('merged_additional_considerations', []) or []:
            if c and c not in items:
                items.append(c)
        for q in compiled.get('unified_questions') or compiled.get('unified_questions_for_round3') or []:
            for ev in q.get('extreme_views') or q.get('extreme_views_with_reasons') or []:
                if ev and ev not in items:
                    items.append(ev)
    if not items:
        return '(none)'
    return '\n'.join((f'- {x}' for x in items))

def run_final_synthesis(case_text: str, consensus: Dict[str, Any], final_outputs: List[Dict[str, Any]], rounds_aggregate_ratings: Dict[str, float], organizer_after_r1: Dict[str, Any], organizer_after_r2: Optional[Dict[str, Any]], case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    organizer = next((p for p in case_participants if p['id'] == ORGANIZER_PARTICIPANT_ID), None)
    if organizer is None:
        raise RuntimeError(f'Organizer {ORGANIZER_PARTICIPANT_ID!r} missing in case participants.')
    rounds_summary = _format_rounds_summary(organizer_after_r1, organizer_after_r2)
    final_answers_block = _format_final_answers_block(final_outputs)
    ratings_block = _format_ratings_block(final_outputs, rounds_aggregate_ratings)
    additional_considerations_block = _format_additional_considerations(organizer_after_r1, organizer_after_r2)
    system_text = build_final_synthesis_system_prompt(organizer['role_name'], organizer['role_description'])
    user_text = build_final_synthesis_user_prompt(case_text=case_text, consensus_emergency=consensus['consensus_emergency_type'], consensus_subclinical=consensus['consensus_subclinical'], rounds_summary=rounds_summary, final_answers_block=final_answers_block, ratings_block=ratings_block, additional_considerations_block=additional_considerations_block, author_id='delphi_consensus')
    response = call_llm(provider=organizer['provider'], model_uri=organizer['model_uri'], system_text=system_text, user_text=user_text, temperature=TEMPERATURE_FINAL_SYNTHESIS, max_tokens=MAX_TOKENS_FINAL_SYNTHESIS)
    print('DELPHI FINAL SYNTHESIS')
    return {'organizer_id': organizer['id'], 'organizer_model_uri': organizer['model_uri'], 'organizer_provider': organizer['provider'], 'organizer_role_name': organizer['role_name'], 'organizer_role_description': organizer['role_description'], 'system_prompt': system_text, 'user_prompt': user_text, 'final_protocol': response['text'].strip(), 'final_emergency_type': extract_emergency_type(response['text']), 'final_subclinical': extract_subclinical(response['text']), 'prompt_tokens': response['prompt_tokens'], 'completion_tokens': response['completion_tokens'], 'total_tokens': response['total_tokens'], 'cached': response['cached']}

def run_delphi_for_case(case_text: str, case_participants: List[Dict[str, str]]) -> Dict[str, Any]:
    round1_outputs = run_round1(case_text, case_participants)
    organizer_after_r1 = run_organizer_compile_after_r1(case_text, round1_outputs, case_participants)
    rounds_executed = 1
    final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round1_outputs]
    organizer_after_r2: Optional[Dict[str, Any]] = None
    early_stop_reason: Optional[str] = None
    round2_outputs: List[Dict[str, Any]] = []
    round3_outputs: List[Dict[str, Any]] = []
    if EARLY_STOP_ON_UNANIMITY and _classifications_unanimous([o['parsed'] for o in round1_outputs]):
        early_stop_reason = 'unanimous_after_r1'
    else:
        organizer_payload = _format_organizer_payload_for_expert(organizer_after_r1['compiled'], 'after_r1')
        round2_outputs = run_round_n(case_text, organizer_payload, 2, case_participants)
        rounds_executed = 2
        final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round2_outputs]
        if EARLY_STOP_ON_UNANIMITY and _classifications_unanimous([o['parsed'] for o in round2_outputs]):
            early_stop_reason = 'unanimous_after_r2'
        elif MAX_DELPHI_ROUNDS >= 3:
            organizer_after_r2 = run_organizer_compile_after_r2(case_text, organizer_after_r1['compiled'], round2_outputs, case_participants)
            organizer_payload_r3 = _format_organizer_payload_for_expert(organizer_after_r2['compiled'], 'after_r2')
            round3_outputs = run_round_n(case_text, organizer_payload_r3, 3, case_participants)
            rounds_executed = 3
            final_round_outputs = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round3_outputs]
    consensus = compute_delphi_consensus(final_round_outputs)
    rounds_aggregate_ratings = _aggregate_ratings(round3_outputs) if round3_outputs else {}
    if round3_outputs:
        final_outputs_for_synth = round3_outputs
    elif round2_outputs:
        final_outputs_for_synth = [{'participant_id': o['participant_id'], 'parsed': o['parsed']} for o in round2_outputs]
    else:
        final_outputs_for_synth = _r1_outputs_as_answer_shape(round1_outputs)
    final_synth = run_final_synthesis(case_text=case_text, consensus=consensus, final_outputs=final_outputs_for_synth, rounds_aggregate_ratings=rounds_aggregate_ratings, organizer_after_r1=organizer_after_r1, organizer_after_r2=organizer_after_r2, case_participants=case_participants)
    return {'round1_outputs': round1_outputs, 'organizer_after_r1': organizer_after_r1, 'round2_outputs': round2_outputs, 'organizer_after_r2': organizer_after_r2, 'round3_outputs': round3_outputs, 'consensus': consensus, 'ratings_aggregate': rounds_aggregate_ratings, 'rounds_executed': rounds_executed, 'early_stop_reason': early_stop_reason, 'final_synthesis': final_synth}


In [ ]:
from pathlib import Path
PANEL_DIR = Path('../delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer').resolve()
PANEL_CSV = PANEL_DIR / 'delphi_noroles.csv'
OUT_R1 = 'delphi_base_noroles_protocol_r1.csv'
OUT_R2 = 'delphi_base_noroles_protocol_r2.csv'
assert PANEL_CSV.exists(), f'нет панельного CSV: {PANEL_CSV}'
panel = pd.read_csv(PANEL_CSV, sep=';', encoding='utf-8-sig', low_memory=False)
panel.columns = [c.lstrip('\ufeff') for c in panel.columns]
panel = panel[panel['Vignette'].notna()].reset_index(drop=True)
PIDS = [p['id'] for p in PARTICIPANTS]
print(f'кейсов: {len(panel)} | участники: {PIDS}')
print('раунды:', panel['delphi_rounds_executed'].value_counts().sort_index().to_dict())

def _outputs_from_raw(row, round_idx):
    outs = []
    for pid in PIDS:
        col = f'r{round_idx}_raw__{pid}'
        raw = row.get(col)
        if raw is None or (isinstance(raw, float) and pd.isna(raw)) or (not str(raw).strip()):
            return None
        parsed = parse_round1_output(str(raw)) if round_idx == 1 else parse_round_n_output(str(raw), round_idx)
        outs.append({'participant_id': pid, 'parsed': parsed, 'raw_text': str(raw)})
    return outs

def _organizer_from_raw(row):
    raw = row.get('organizer_after_r1_raw')
    if raw is None or (isinstance(raw, float) and pd.isna(raw)) or (not str(raw).strip()):
        return None
    return {'compiled': parse_compile_after_r1_output(str(raw)), 'raw_text': str(raw)}

def synthesize_for_round(row, target_round):
    executed = int(row['delphi_rounds_executed'])
    if executed <= target_round:
        return (str(row['winner_protocol']), 'reused_final', 0, str(row.get('winner_final_emergency_type', '')), str(row.get('winner_final_subclinical', '')))
    outs = _outputs_from_raw(row, target_round)
    org1 = _organizer_from_raw(row)
    if outs is None or org1 is None:
        return ('', 'parse_failed', 0, '', '')
    consensus = compute_delphi_consensus(outs)
    final_outputs = _r1_outputs_as_answer_shape(outs) if target_round == 1 else outs
    synth = run_final_synthesis(case_text=row['case_text'], consensus=consensus, final_outputs=final_outputs, rounds_aggregate_ratings={}, organizer_after_r1=org1, organizer_after_r2=None, case_participants=[dict(p) for p in PARTICIPANTS])
    return (synth['final_protocol'], 'generated', synth['total_tokens'], synth['final_emergency_type'], synth['final_subclinical'])
KEEP = ['Diagnosis10cm', 'Diagnosis10', 'Diagnosis_vulgar', 'Vignette', 'Review of Systems', 'Past Medical History', 'Physical Examination', 'Test Results', 'Simplified (added symptoms)', 'Type', 'Vignette_race', 'Genealogy', 'Sub Clinical Specialties', 'Age Group', 'First Role', 'Second Role', 'Third Role', 'Criterion', 'Criterion_uni', 'case_text', 'delphi_rounds_executed', 'delphi_early_stop_reason']
RUN_SYNTH = True
if RUN_SYNTH:
    for target_round, out_csv in [(1, OUT_R1), (2, OUT_R2)]:
        recs = []
        for _, row in tqdm(panel.iterrows(), total=len(panel), desc=f'synth@R{target_round}'):
            prot, src, tok, em, sub = synthesize_for_round(row, target_round)
            rec = {k: row.get(k) for k in KEEP if k in panel.columns}
            rec['winner_protocol'] = prot
            rec['protocol_round'] = target_round
            rec['protocol_source'] = src
            rec['synth_total_tokens'] = tok
            rec['protocol_emergency_type'] = em
            rec['protocol_subclinical'] = sub
            recs.append(rec)
        out = pd.DataFrame(recs)
        out.to_csv(out_csv, sep=';', index=False, encoding='utf-8-sig')
        print(f"\n{out_csv}: {len(out)} строк | {out['protocol_source'].value_counts().to_dict()} | синтез {out['synth_total_tokens'].sum() / 1000000.0:.2f}M токенов")
else:
    print('RUN_SYNTH=False — синтез пропущен')
